In [1]:
!pip install -q timm albumentations==1.3.1 opencv-python pyyaml tqdm
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -q -r requirements.txt


fatal: destination path 'yolov5' already exists and is not an empty directory.
/content/yolov5


In [2]:
!wget -q https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt
!ls -lh yolov5s.pt


-rw-r--r-- 1 root root 15M Nov 22  2022 yolov5s.pt


In [3]:
!pip install roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="GYiLvqoCXhBZ3NWSGyrR")
project = rf.workspace("modd2").project("pascal-voc-2007-bhjgf")
dataset = project.version(1).download("yolov5")


loading Roboflow workspace...
loading Roboflow project...


In [4]:
import yaml, os
from pathlib import Path

DATA_YAML = "/content/yolov5/Pascal-Voc-2007-1/data.yaml"

with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

print(data)
print("\ntrain path exists:", Path(data["train"]).exists())
print("val path exists:", Path(data["val"]).exists())


{'names': ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'], 'nc': 20, 'roboflow': {'license': 'CC BY 4.0', 'project': 'pascal-voc-2007-bhjgf', 'url': 'https://universe.roboflow.com/project/pascal-voc-2007-bhjgf/dataset/1', 'version': 1, 'workspace': 'project'}, 'test': '../test/images', 'train': 'Pascal-Voc-2007-1/train/images', 'val': 'Pascal-Voc-2007-1/valid/images'}

train path exists: True
val path exists: True


In [5]:
%%writefile swin_yolov5_pascal2007_ca_fs_p3_patch_embedB.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import timm

from models.yolo import Detect


# ------------------------------------------------------------
# COORDINATE ATTENTION (CA)
# ------------------------------------------------------------
class CoordinateAttention(nn.Module):
    def __init__(self, inp, reduction=32):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, inp // reduction)

        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))

        out = identity * a_h * a_w
        return out


# ------------------------------------------------------------
# FEATURE SELECTION (Minimal + Residual)
# ------------------------------------------------------------
class FeatureSelector(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 8)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(hidden, channels, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        w = self.gap(x)
        w = self.fc1(w)
        w = self.act(w)
        w = self.fc2(w)
        w = self.sigmoid(w)
        return x + (x * w)


# ------------------------------------------------------------
# SWIN BACKBONE (Patch Embedding Option B)
# ------------------------------------------------------------
class SwinBackbone(nn.Module):
    def __init__(self, variant="swinv2_tiny_window8_256"):
        super().__init__()

        self.swin = timm.create_model(
            variant,
            pretrained=True,
            num_classes=0,
        )

        self.cv3 = nn.Conv2d(192, 256, 1)
        self.cv4 = nn.Conv2d(384, 512, 1)
        self.cv5 = nn.Conv2d(768, 1024, 1)

        self.ca_p3 = CoordinateAttention(256)
        self.fs_p3 = FeatureSelector(256)

        self.required_multiple = 8 * 16

    def forward(self, x):
        B, C, H0, W0 = x.shape
        m = self.required_multiple

        pad_h = (math.ceil(H0 / m) * m) - H0
        pad_w = (math.ceil(W0 / m) * m) - W0
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))

        Hp, Wp = x.shape[-2], x.shape[-1]

        # ------------------------------------------------------------
        # PATCH EMBEDDING (Explicit, pretrained weights reused)
        # ------------------------------------------------------------
        x = self.swin.patch_embed(x)

        # Some timm models have absolute_pos_embed (mostly SwinV1)
        if hasattr(self.swin, "absolute_pos_embed") and self.swin.absolute_pos_embed is not None:
            x = x + self.swin.absolute_pos_embed

        # SwinV2 usually has no pos_drop, but some variants do
        if hasattr(self.swin, "pos_drop") and self.swin.pos_drop is not None:
            x = self.swin.pos_drop(x)


        # ------------------------------------------------------------
        # STAGES
        # We compute resolution explicitly (NO guessing)
        # ------------------------------------------------------------
        def tokens_to_map(t, H, W):
            # t: (B, L, C), where L == H*W
            return t.view(B, H, W, t.shape[-1]).permute(0, 3, 1, 2).contiguous()

        # After patch_embed: H/4, W/4
        Ht, Wt = Hp // 4, Wp // 4

        outs = []
        for i, layer in enumerate(self.swin.layers):
            x = layer(x)

            # After stage0 -> still H/4
            # After stage1 -> H/8
            # After stage2 -> H/16
            # After stage3 -> H/32
            if i > 0:
                Ht //= 2
                Wt //= 2

            # Collect stage1, stage2, stage3 outputs
            if i in (1, 2, 3):
                outs.append(tokens_to_map(x, Ht, Wt))

        f3, f4, f5 = outs  # f3: H/8, f4: H/16, f5: H/32

        p3 = self.cv3(f3)
        p4 = self.cv4(f4)
        p5 = self.cv5(f5)

        p3 = self.ca_p3(p3)
        p3 = self.fs_p3(p3)

        return [p3, p4, p5]


# ------------------------------------------------------------
# SWIN + YOLOv5 DETECT
# ------------------------------------------------------------
class SwinYOLO(nn.Module):
    def __init__(self, nc=20, anchors=None):
        super().__init__()

        self.backbone = SwinBackbone()

        self.detect = Detect(
            nc=nc,
            anchors=anchors,
            ch=[256, 512, 1024],
        )

        self.detect.stride = torch.tensor([8., 16., 32.])

        self.model = nn.ModuleList([self.backbone, self.detect])
        self.nc = nc
        self.names = [str(i) for i in range(nc)]

    def forward(self, x, augment=False):
      # Ensure anchors are always on same device as x
      if self.detect.anchors.device != x.device:
          self.detect.anchors = self.detect.anchors.to(x.device)

      feats = self.backbone(x)
      return self.detect(feats)



# ------------------------------------------------------------
# LOAD PRETRAINED YOLOv5 DETECT WEIGHTS
# ------------------------------------------------------------
def load_yolov5_detect(model, weights="yolov5s.pt"):
    ckpt = torch.load(weights, map_location="cpu", weights_only=False)
    state = ckpt["model"].float().state_dict()

    k_anchors = "model.24.anchors"
    if k_anchors in state:
        model.detect.anchors = state[k_anchors].clone()
        print("✅ Loaded pretrained anchors into Detect")
    else:
        print("⚠️ Could not find anchors in checkpoint")

    print("Detect nc:", model.detect.nc)
    print("Detect anchors:", model.detect.anchors)
    print("Detect stride:", model.detect.stride)

    # SAFE (works in YOLOv5 v7.0)
    if hasattr(model.detect, "m"):
        print("Detect input channels:", [m.in_channels for m in model.detect.m])
    else:
        print("⚠️ Detect has no attribute 'm' (unexpected in YOLOv5)")



Overwriting swin_yolov5_pascal2007_ca_fs_p3_patch_embedB.py


In [6]:
import torch
ckpt = torch.load("/content/yolov5/yolov5s.pt", map_location="cpu", weights_only=False)
state = ckpt["model"].float().state_dict()

detect_keys = [k for k in state.keys() if ".anchors" in k or k.endswith(".m.0.weight")]
print("Example detect-like keys:")
for k in detect_keys[:30]:
    print(k)


Example detect-like keys:
model.24.anchors
model.24.m.0.weight


In [7]:
import torch
from swin_yolov5_pascal2007_ca_fs_p3_patch_embedB import SwinYOLO, load_yolov5_detect

anchors = [
    [10,13, 16,30, 33,23],
    [30,61, 62,45, 59,119],
    [116,90, 156,198, 373,326],
]

model = SwinYOLO(nc=20, anchors=anchors)
load_yolov5_detect(model)

x = torch.zeros(1, 3, 256, 256)
y = model(x)

for i, out in enumerate(y):
    print(f"Scale {i} output shape:", out.shape)

loss = sum([o.sum() for o in y])
loss.backward()
print("\n✅ Backward pass successful")


✅ Loaded pretrained anchors into Detect
Detect nc: 20
Detect anchors: tensor([[[ 1.25000,  1.62500],
         [ 2.00000,  3.75000],
         [ 4.12500,  2.87500]],

        [[ 1.87500,  3.81250],
         [ 3.87500,  2.81250],
         [ 3.68750,  7.43750]],

        [[ 3.62500,  2.81250],
         [ 4.87500,  6.18750],
         [11.65625, 10.18750]]])
Detect stride: tensor([ 8., 16., 32.])
Detect input channels: [256, 512, 1024]
Scale 0 output shape: torch.Size([1, 3, 32, 32, 25])
Scale 1 output shape: torch.Size([1, 3, 16, 16, 25])
Scale 2 output shape: torch.Size([1, 3, 8, 8, 25])

✅ Backward pass successful


In [10]:
%%writefile train_swin_yolo_ca_fs_p3_patch_embedB.py
import torch
import yaml
from pathlib import Path
from tqdm import tqdm
import sys
from copy import deepcopy
import csv

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
YOLO_ROOT = Path("/content/yolov5")
DATA_YAML = "/content/yolov5/Pascal-Voc-2007-1/data.yaml"
HYP_YAML = "/content/yolov5/data/hyps/hyp.scratch-low.yaml"

sys.path.append(str(YOLO_ROOT))

# ------------------------------------------------------------
# YOLOv5 IMPORTS
# ------------------------------------------------------------
from utils.dataloaders import create_dataloader
from utils.loss import ComputeLoss
from val import run as val_run
from utils.torch_utils import select_device

# ------------------------------------------------------------
# OUR MODEL
# ------------------------------------------------------------
from swin_yolov5_pascal2007_ca_fs_p3_patch_embedB import SwinYOLO, load_yolov5_detect

# ------------------------------------------------------------
# TRAINING CONFIG
# ------------------------------------------------------------
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 40

LR = 2e-4
FREEZE_EPOCHS = 5

DEVICE = select_device("0" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# LOAD DATA + HYP
# ------------------------------------------------------------
with open(DATA_YAML) as f:
    data = yaml.safe_load(f)

with open(HYP_YAML) as f:
    hyp = yaml.safe_load(f)

nc = int(data["nc"])

# ------------------------------------------------------------
# BUILD MODEL
# ------------------------------------------------------------
anchors = [
    [10, 13, 16, 30, 33, 23],
    [30, 61, 62, 45, 59, 119],
    [116, 90, 156, 198, 373, 326],
]

model = SwinYOLO(nc=nc, anchors=anchors).to(DEVICE)
load_yolov5_detect(model)

# 🔥 CRITICAL: move anchors tensor to same device
model.detect.anchors = model.detect.anchors.to(DEVICE)


model.hyp = hyp
model.train()

# ------------------------------------------------------------
# FREEZE BACKBONE
# ------------------------------------------------------------
for p in model.backbone.parameters():
    p.requires_grad = False

print(f"🔒 Swin backbone frozen for first {FREEZE_EPOCHS} epochs")

# ------------------------------------------------------------
# DATALOADERS
# ------------------------------------------------------------
train_loader, _ = create_dataloader(
    path=data["train"],
    imgsz=IMG_SIZE,
    batch_size=BATCH_SIZE,
    stride=32,
    single_cls=False,
    hyp=hyp,
    augment=True,
    cache=False,
    rect=False,
    rank=-1,
    workers=4,
)

val_loader, _ = create_dataloader(
    path=data["val"],
    imgsz=IMG_SIZE,
    batch_size=BATCH_SIZE,
    stride=32,
    single_cls=False,
    hyp=hyp,
    augment=False,
    cache=False,
    rect=True,
    rank=-1,
    workers=4,
)

# ------------------------------------------------------------
# LOSS + OPTIMIZER
# ------------------------------------------------------------
compute_loss = ComputeLoss(model)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay=5e-4
)

# ------------------------------------------------------------
# CSV LOGGING
# ------------------------------------------------------------
csv_path = "/content/swin_yolov5_ca_fs_p3_patch_embedB_training_metrics.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch",
        "box_loss", "obj_loss", "cls_loss", "total_loss",
        "precision", "recall", "map50", "map5095"
    ])

# ------------------------------------------------------------
# TRAIN LOOP
# ------------------------------------------------------------
for epoch in range(EPOCHS):

    if epoch == FREEZE_EPOCHS:
        print("🔓 Unfreezing Swin backbone")
        for p in model.backbone.parameters():
            p.requires_grad = True

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    mloss = torch.zeros(3, device=DEVICE)

    for imgs, targets, paths, _ in pbar:
        imgs = imgs.to(DEVICE).float() / 255.0
        targets = targets.to(DEVICE)

        preds = model(imgs)
        loss, loss_items = compute_loss(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()

        mloss = mloss * 0.9 + loss_items * 0.1

        pbar.set_postfix(
            box=f"{mloss[0]:.3f}",
            obj=f"{mloss[1]:.3f}",
            cls=f"{mloss[2]:.3f}",
            total=f"{loss.item():.3f}",
        )

    print(f"Epoch {epoch+1}/{EPOCHS} | loss: {loss.item():.4f}")

    # -------------------- VALIDATION --------------------
    model.eval()

    with torch.no_grad():
        eval_model = deepcopy(model).to(DEVICE)

        results, maps, _ = val_run(
            data=data,
            model=eval_model,
            dataloader=val_loader,
            imgsz=IMG_SIZE,
            conf_thres=0.001,
            iou_thres=0.6,
            device=DEVICE,
            single_cls=False,
            save_json=False,
            verbose=False,
        )

    del eval_model
    torch.cuda.empty_cache()

    model.train()
    torch.set_grad_enabled(True)

    P, R, mAP50, mAP5095 = results[:4]

    print(
        f"📊 Val | "
        f"P: {P:.4f} | "
        f"R: {R:.4f} | "
        f"mAP@0.5: {mAP50:.4f} | "
        f"mAP@0.5:0.95: {mAP5095:.4f}"
    )

    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch + 1,
            float(mloss[0].item()),
            float(mloss[1].item()),
            float(mloss[2].item()),
            float(loss.item()),
            float(P),
            float(R),
            float(mAP50),
            float(mAP5095)
        ])

print("\n✅ Training complete")
print(f"✅ CSV saved to: {csv_path}")


Overwriting train_swin_yolo_ca_fs_p3_patch_embedB.py


In [11]:
!python train_swin_yolo_ca_fs_p3_patch_embedB.py


YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

✅ Loaded pretrained anchors into Detect
Detect nc: 20
Detect anchors: tensor([[[ 1.25000,  1.62500],
         [ 2.00000,  3.75000],
         [ 4.12500,  2.87500]],

        [[ 1.87500,  3.81250],
         [ 3.87500,  2.81250],
         [ 3.68750,  7.43750]],

        [[ 3.62500,  2.81250],
         [ 4.87500,  6.18750],
         [11.65625, 10.18750]]])
Detect stride: tensor([ 8., 16., 32.])
Detect input channels: [256, 512, 1024]
🔒 Swin backbone frozen for first 5 epochs
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))
Scanning Pascal-Voc-2007-1/train/labels.cache... 3506 images, 32 backgrounds, 0 corrupt: 100% 3506/3506 [00:00<?, ?it/s]
Scanning Pascal-Voc-2007-1/valid/labels.cache... 1001 images, 6 backgrounds, 0 corrupt: 100% 1001/1001 [00:00<?, ?it/s]
Epoch 1/40: 100% 439/439 [00:54